In [ ]:
%pip install numpy matplotlib scipy > /dev/null
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import kalman
kalman.set_debug(0)

import kalman.runtime.chrono as chrono
import numpy as np
import matplotlib.pyplot as plt

# Power Grid Harmonic Tracking

## Why track power grid harmonics?

A power grid delivers AC voltage at a nominal fundamental frequency — 50 Hz in Europe 
and most of Asia, 60 Hz in the Americas.  In an ideal world that voltage would be a 
perfect sinusoid.  In the real world, **non-linear loads** (rectifiers, motor drives, 
LED drivers, arc furnaces, EV chargers) draw current in bursts rather than continuously, 
injecting **harmonic distortion** at multiples of the fundamental.

**Why it matters:**
- Harmonic currents cause transformer overheating, cable losses, and resonance overvoltages.
- Standards (IEC 61000-4-7, IEEE 519) mandate that voltage THD stays below 5 % at the point 
  of common coupling.
- Active harmonic filters must estimate each harmonic's amplitude and phase in real time 
  to inject a cancelling current.

Traditional DFT-based meters suffer from spectral leakage when the fundamental drifts 
and can only update once per FFT window.  The Kalman approach in this notebook tracks 
every harmonic *continuously*, quantifies its uncertainty, and is causal (streaming).

---
## 1  The phasor — the engineer's tool

Every electrical engineer describes a sinusoid by its **phasor**: a single complex number 
that encodes the amplitude and phase simultaneously.  For a signal

$$v(t) = V \cos(2\pi f_0 t + \varphi)$$

the phasor is $\mathbf{V} = V e^{i\varphi} = V\cos\varphi + i\,V\sin\varphi$, typically 
drawn as an arrow in the complex plane.

**Why phasors?**  Because circuit analysis becomes algebraic:
- Ohm's law: $\mathbf{V} = \mathbf{I} \cdot \mathbf{Z}$ (complex impedance)
- Power: $P = \tfrac{1}{2} \mathrm{Re}[\mathbf{V} \mathbf{I}^*]$ (active power)
- Superposition: $v_1(t) + v_2(t) \leftrightarrow \mathbf{V}_1 + \mathbf{V}_2$ (vector addition)

For a distorted grid voltage with multiple harmonics, each harmonic $k$ has its own phasor 
$\mathbf{V}_k = \mathrm{Re}_k + i\,\mathrm{Im}_k$, and the time-domain signal is the 
real part of the rotating sum:

$$v(t) = \sum_{k=1}^{N} \mathrm{Re}_k \cos(\omega_k t) - \mathrm{Im}_k \sin(\omega_k t), 
\qquad \omega_k = 2\pi k f_0$$

In [ ]:
# Visualise phasors in the complex plane and their time-domain reconstruction
f0, fs = 50.0, 1024.0
T = 1.0 / f0         # one fundamental period
t_cyc = np.linspace(0, T, 500)

# Three phasors: fundamental (230 V), 3rd harmonic (15 V), 5th harmonic (5 V)
phasors = [(230.0, 0.0), (15.0, np.pi / 4), (5.0, np.pi / 3)]
harmonics_labels = ["H1 (f₀)", "H3 (3f₀)", "H5 (5f₀)"]
colors = ["steelblue", "darkorange", "seagreen"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Power grid phasors — amplitude, phase, and time-domain reconstruction", fontweight="bold")

# Phasor diagram
ax = axes[0]
ax.set_aspect("equal")
for (V, phi), lbl, col in zip(phasors, harmonics_labels, colors):
    ax.annotate("", xy=(V * np.cos(phi), V * np.sin(phi)), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=col, lw=2))
    ax.text(V * np.cos(phi) * 1.08, V * np.sin(phi) * 1.08,
            f"{lbl}\nV={V}V, φ={np.degrees(phi):.0f}°", color=col, fontsize=7.5)
ax.set_xlim(-260, 260); ax.set_ylim(-50, 50)
ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("Re [V]"); ax.set_ylabel("Im [V]")
ax.set_title("Phasor diagram (complex plane)")

# Individual harmonic waveforms
ax = axes[1]
for k, ((V, phi), lbl, col) in enumerate(zip(phasors, harmonics_labels, colors)):
    fk = (2 * k + 1) * f0   # odd harmonics: H1, H3, H5
    ax.plot(t_cyc * 1e3, V * np.cos(2 * np.pi * fk * t_cyc + phi), color=col, lw=1.5, label=lbl)
ax.set_xlabel("t [ms]"); ax.set_ylabel("voltage [V]")
ax.set_title("Individual harmonics")
ax.axhline(0, color="gray", lw=0.5); ax.legend(fontsize=8)

# Composite waveform
ax = axes[2]
v_clean = sum(V * np.cos(2 * np.pi * (2*k+1) * f0 * t_cyc + phi)
              for k, (V, phi) in enumerate(phasors))
ax.plot(t_cyc * 1e3, v_clean, color="purple", lw=2, label="H1 + H3 + H5")
ax.plot(t_cyc * 1e3, phasors[0][0] * np.cos(2 * np.pi * f0 * t_cyc), "--",
        color="steelblue", lw=1, label="H1 only (ideal)")
ax.set_xlabel("t [ms]"); ax.set_ylabel("voltage [V]")
ax.set_title("Composite: distorted vs ideal")
ax.axhline(0, color="gray", lw=0.5); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

---
## 2  Why only odd harmonics?

An ideal AC waveform has **half-wave symmetry**: `v(t + T/2) = −v(t)`.  This means the 
positive and negative half-cycles are mirror images.  A standard Fourier argument shows 
that any half-wave-symmetric signal has **only odd harmonics** in its Fourier series — 
the even coefficients are exactly zero.

**Physical origin:**
- A transformer, resistive load, or motor drive that is symmetric in both half-cycles 
  preserves half-wave symmetry → only H1, H3, H5, H7, …
- A **half-wave rectifier** clips one half-cycle → breaks symmetry → injects H2, H4, …
- **DC bias** (geomagnetic induced current in a transformer) also breaks symmetry → even harmonics.
- Unbalanced three-phase loading: even harmonics on the neutral.

**Diagnostic value:** Tracking all harmonics (including H2, H4) and observing that 
the even ones are ≈ 0 is a **physics verification**.  A non-zero even harmonic is an 
immediate fault indicator — something has broken the half-wave symmetry.

In [ ]:
from numpy.fft import rfft, rfftfreq

fs_demo = 1024.0; T_demo = 4.0
t_d = np.arange(int(T_demo * fs_demo)) / fs_demo
V1, V3, V5 = 230.0, 15.0, 5.0

# Ideal: half-wave symmetric (odd only)
v_odd  = (V1 * np.cos(2*np.pi*f0*t_d) +
          V3 * np.cos(2*np.pi*3*f0*t_d + np.pi/4) +
          V5 * np.cos(2*np.pi*5*f0*t_d + np.pi/3))

# Faulty: half-wave rectifier adds H2 (2nd harmonic)
V2_fault = 20.0
v_fault = v_odd + V2_fault * np.cos(2*np.pi*2*f0*t_d + 0.8)

fr_d = rfftfreq(len(t_d), 1/fs_demo)
norm = 2 / len(t_d)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Half-wave symmetry: ideal (odd only) vs fault (even harmonics appear)", fontweight="bold")

for ax, v, title, col in zip(axes,
                              [v_odd, v_fault],
                              ["Ideal (symmetric): H2, H4 = 0", "Half-wave rectifier fault: H2 ≠ 0"],
                              ["steelblue", "crimson"]):
    spec = np.abs(rfft(v)) * norm
    ax.bar(fr_d[(fr_d < 350)], spec[(fr_d < 350)], width=0.3, color=col, alpha=0.7)
    for k in range(1, 7):
        ax.axvline(k * f0, ls="--", color="gray", lw=0.8, alpha=0.5)
        ax.text(k * f0, spec.max() * 0.02, f"H{k}", ha="center", fontsize=8, color="gray")
    ax.set_xlim(0, 340); ax.set_xlabel("Frequency [Hz]"); ax.set_ylabel("|V| [V]")
    ax.set_title(title)

plt.tight_layout(); plt.show()
print("Ideal grid:  H1=%.0f V,  H2=%.1f V,  H3=%.0f V  (H2 ≈ 0 ✓)" % (
    np.abs(rfft(v_odd)[int(f0*T_demo)])*norm,
    np.abs(rfft(v_odd)[int(2*f0*T_demo)])*norm,
    np.abs(rfft(v_odd)[int(3*f0*T_demo)])*norm))
print("Fault grid:  H1=%.0f V,  H2=%.1f V,  H3=%.0f V  (H2 ≠ 0 — rectifier fault)" % (
    np.abs(rfft(v_fault)[int(f0*T_demo)])*norm,
    np.abs(rfft(v_fault)[int(2*f0*T_demo)])*norm,
    np.abs(rfft(v_fault)[int(3*f0*T_demo)])*norm))

---
## 3  The Kalman filter as a phasor meter

### Lab-frame formulation — no demodulation

We want to track the phasor of each harmonic from the **raw measured voltage** — no 
Hilbert transform, no lock-in demodulation.  This is the **lab-frame** approach 
(contrast with notebook 02's rotating-frame `OscillatorModel` that demodulates first).

**State:** all phasors stacked, one per harmonic:
$$x = [\mathrm{Re}_1, \mathrm{Im}_1,\ \mathrm{Re}_2, \mathrm{Im}_2,\ \dots,\ \mathrm{Re}_N, \mathrm{Im}_N]^\top \quad (2N \text{ states})$$

**Dynamics:** $x_{k+1} = x_k + w_k$ with $F = I$ — the phasors of a steady sinusoid 
are constant.  Process noise $w_k$ (small `process_uncertainty`) allows slow drift.

**Observation:** the scalar voltage is the real part of the rotating sum — the carrier 
rotation lives in the **time-varying observation map**:
$$z(t) = v(t) = \sum_k \mathrm{Re}_k \cos(\omega_k t) - \mathrm{Im}_k \sin(\omega_k t)$$
$$H(t) = [\cos\omega_1 t,\ -\sin\omega_1 t,\ \cos\omega_2 t,\ -\sin\omega_2 t,\ \dots] \quad (1 \times 2N)$$

### Why joint estimation prevents leakage

Because all $N$ phasors must **together** explain the single measured voltage, the 
strong fundamental H1 (230 V) is fully claimed by its own phasor.  An absent harmonic 
(say H2) is therefore forced to zero — there is no leftover signal for it to absorb.

A bank of *independent* per-harmonic demodulators cannot do this: the 230 V fundamental 
survives the H2 demodulation as a constant-magnitude rotating phasor and shows up as a 
spurious ~23 V "amplitude" even when H2 is truly zero.

### Lab frame vs Rotating frame — where the carrier sits

| | Lab frame (`PhasorOscillatorModel`) | Rotating frame (`OscillatorModel`, nb 02) |
|---|---|---|
| Carrier lives in | `H(t)` (time-varying obs. map) | `F = expm(Δω)` (state transition) |
| `F` | `I` | `expm(Δω·dt)` |
| `H` | `[cos ωt, −sin ωt]` (time-varying) | identity |
| Demodulation | none — real signal in | Hilbert → baseband |
| Lines tracked | any set, wide-band | one narrow window per mode |
| Best for | strong, wide-spread (mains) | weak, narrowband (violin modes) |

---
## 4  `PhasorOscillatorModel` — generic lab-frame tracker

The base class tracks **any set of frequencies** (not necessarily harmonics) from a 
**real signal** — no demodulation.  Feed it any two unrelated tones and it recovers both.

In [ ]:
from kalman.apps.oscillators import PhasorOscillatorModel

fsd, Td = 1024.0, 4.0
td = np.arange(int(Td * fsd)) / fsd
f_a, f_b = 60.0, 173.2
A_a, A_b = 1.0, 0.5
sig = (A_a * np.cos(2 * np.pi * f_a * td + 0.4)
       + A_b * np.cos(2 * np.pi * f_b * td - 1.1)
       + 0.05 * np.random.randn(len(td)))

phasor = PhasorOscillatorModel("demo", [f_a, f_b], fs=fsd,
                               measurement_uncertainty=0.05,
                               process_uncertainty=1e-4, p0_std=10.0)
rec_p = phasor.filter(sig, smooth=True)
amp_p = phasor.amplitudes(rec_p)
recon_p = phasor.reconstruct(rec_p)

print(f"n={phasor.n}, m={phasor.m}  (2 phasors → 4 states, 1 real channel — no demodulation)")
print("Recovered amplitudes (steady-state, want %.2f, %.2f): %s" %
      (A_a, A_b, np.round(amp_p[len(td)//2:].mean(axis=0), 3)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(td, amp_p[:, 0], color="darkorange", label=f"|ψ| @ {f_a} Hz")
axes[0].plot(td, amp_p[:, 1], color="navy", label=f"|ψ| @ {f_b} Hz")
axes[0].axhline(A_a, ls="--", color="gray"); axes[0].axhline(A_b, ls="--", color="gray")
axes[0].set_title("Lab-frame phasor amplitudes (no demodulation)")
axes[0].set_xlabel("t [s]"); axes[0].set_ylabel("|ψ|"); axes[0].legend()
ns = 300
axes[1].plot(td[:ns], sig[:ns], color="gray", alpha=0.5, label="measured (real)")
axes[1].plot(td[:ns], recon_p[:ns], color="darkorange", lw=1.2, label="reconstructed")
axes[1].set_title("Waveform — real in, real out"); axes[1].set_xlabel("t [s]"); axes[1].legend()
plt.tight_layout(); plt.show()

---
## 5  Synthetic power grid signal

We build a **controlled test signal** with known harmonic content — fundamental (50 Hz, 230 V), 
3rd (150 Hz, 15 V), 5th (250 Hz, 5 V), and Gaussian noise (σ = 10 V).  
The even harmonics H2, H4 are deliberately absent (ideal symmetric load).  
We then track five harmonics and expect H2/H4 to settle at ≈ 0.

In [ ]:
f0   = 50.0; fs = 1024.0; T = 5.0
t    = np.arange(int(T * fs)) / fs

A1, phi1 = 230.0, 0.0
A3, phi3 =  15.0, np.pi / 4
A5, phi5 =   5.0, np.pi / 3

v_clean = (A1 * np.cos(2 * np.pi * f0 * t + phi1)
         + A3 * np.cos(2 * np.pi * 3 * f0 * t + phi3)
         + A5 * np.cos(2 * np.pi * 5 * f0 * t + phi5))
noise_std = 10.0
np.random.seed(0)
v_noisy = v_clean + np.random.normal(0, noise_std, len(t))

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
axes[0].plot(t, v_noisy, alpha=0.5, color="gray", label="Measured (noisy)")
axes[0].plot(t, v_clean, "--", color="steelblue", label="Truth")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Voltage (V)")
axes[0].set_title("Synthetic grid: 50 Hz + H3 + H5 + noise")
axes[0].legend()

freqs_d = rfftfreq(len(t), 1/fs)
spectrum = np.abs(rfft(v_noisy)) * 2 / len(t)
axes[1].semilogy(freqs_d, spectrum, color="gray", alpha=0.7)
for fk, lbl in [(50,'H1'),(150,'H3'),(250,'H5')]:
    axes[1].axvline(fk, ls="--", color="darkorange", lw=1.5, label=f"{lbl}: {fk} Hz")
axes[1].set_xlim(0, 400); axes[1].set_xlabel("Frequency (Hz)"); axes[1].set_ylabel("|FFT|")
axes[1].set_title("Amplitude spectrum — odd harmonics dominate")
axes[1].legend()
plt.tight_layout(); plt.show()

---
## 6  `PowerGridStateModel` — mains specialization

`PowerGridStateModel` is `PhasorOscillatorModel` with the candidate frequencies fixed to 
the harmonic ladder of a fundamental `f0`.  Everything else — F=I, H(t), joint estimation, 
no demodulation — is inherited from the base class.

We track **five harmonics (H1…H5)** simultaneously, seeding H1/H3/H5 with reasonable 
initial values and zeroing H2/H4.  The filter's task:
- Converge H1 → 230 V, H3 → 15 V, H5 → 5 V  
- Keep H2, H4 at ≈ 0  (joint estimation prevents leakage)

In [ ]:
from kalman.apps.powergrids import PowerGridStateModel

chrono.restart()
grid = PowerGridStateModel("L1", f0=f0, harmonics=5, fs=fs,
                           measurement_uncertainty=noise_std,
                           process_uncertainty=0.01)
grid.initial_state([[A1, 0],   # H1
                    [0,  0],   # H2 (absent)
                    [A3, 0],   # H3
                    [0,  0],   # H4 (absent)
                    [A5, 0]])  # H5
grid.print()
chrono.lap("Model initialized")

record = grid.filter(v_noisy, smooth=True, reset=True)
chrono.lap("Filtering completed")

grid.plot(record, measured=v_noisy, truth=v_clean,
          true_amplitudes=[A1, 0, A3, 0, A5])

amplitudes, sigma_amp = grid.harmonic_envelope(record)
chrono.summary()

### Dynamic waveform replay

Slide a 200 ms window to watch the Kalman reconstruction track measured vs truth — 
follow the acquisition transient into steady state.

In [ ]:
grid.plot(record, interactive=True, measured=v_noisy, truth=v_clean, window_ms=200)

---
## 7  Total Harmonic Distortion (THD) with uncertainty

$$\mathrm{THD} = \frac{\sqrt{A_2^2 + A_3^2 + \cdots + A_N^2}}{A_1} \times 100\%$$

Because the Kalman filter maintains a covariance matrix, we can propagate the per-harmonic 
amplitude uncertainty $\sigma_{A_k}$ (from $P_k$) to obtain **±1σ confidence bands on THD** — 
impossible with a plain FFT.

In [ ]:
import matplotlib.ticker as mticker

N_harm = grid.n_harmonics
A1_est = amplitudes[:, 0]
D      = np.sqrt(np.sum(amplitudes[:, 1:]**2, axis=1))
thd    = 100 * D / (A1_est + 1e-12)

sigma_thd = np.zeros(len(t))
denom = (A1_est * D + 1e-12)
for k in range(1, N_harm):
    sigma_thd += (amplitudes[:, k] / denom)**2 * sigma_amp[:, k]**2
sigma_thd += (thd / (A1_est + 1e-12))**2 * sigma_amp[:, 0]**2
sigma_thd = 100 * np.sqrt(sigma_thd)

thd_true = 100 * np.sqrt(A3**2 + A5**2) / A1
ss_slice = slice(-len(t) // 10, None)
ss_amps  = amplitudes[ss_slice].mean(axis=0)
ss_thd   = thd[ss_slice].mean()
true_amps = np.array([A1, 0, A3, 0, A5])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Total Harmonic Distortion — Kalman Estimates", fontweight="bold")

ax = axes[0]
ax.plot(t, thd, color="darkorange", lw=1.5, label="Kalman THD")
ax.fill_between(t, thd - sigma_thd, thd + sigma_thd, color="darkorange", alpha=0.25, label="±1σ")
ax.axhline(thd_true, ls="--", color="steelblue", lw=1.5, label=f"True THD = {thd_true:.2f} %")
ax.axhline(5.0, ls=":", color="red", lw=1.2, alpha=0.7, label="IEC 61000-4-7 limit (5 %)")
ax.set_xlabel("Time (s)"); ax.set_ylabel("THD (%)")
ax.set_title("THD time series (±1σ from filter covariance)")
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

ax = axes[1]
x_pos = np.arange(1, N_harm + 1)
bars_t = ax.bar(x_pos - 0.22, true_amps, 0.40, label="True", color="steelblue", alpha=0.85)
bars_e = ax.bar(x_pos + 0.22, ss_amps,   0.40, label="Kalman (steady-state)", color="darkorange", alpha=0.85)
ax.errorbar(x_pos + 0.22, ss_amps, yerr=sigma_amp[ss_slice].mean(axis=0),
            fmt="none", color="black", capsize=4, lw=1.2, label="±1σ")
for bar, val in zip(bars_e, ss_amps):
    if val > 1:
        ax.text(bar.get_x() + bar.get_width()/2, val + 1.5, f"{val:.1f}", ha="center", fontsize=7.5)
ax.set_xlabel("Harmonic order"); ax.set_ylabel("Amplitude (V)")
ax.set_title("Per-harmonic: truth vs Kalman steady-state")
ax.set_xticks(x_pos); ax.set_xticklabels([f"H{k}" for k in range(1, N_harm + 1)])
ax.grid(True, axis="y", alpha=0.3); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()
print(f"Steady-state THD: {ss_thd:.3f} %  (true: {thd_true:.3f} %)")

---
## 8  Fault diagnosis — half-wave rectifier load

A half-wave rectifier clips one half-cycle of the supply current, breaking half-wave 
symmetry and injecting **even harmonics** (H2, H4, …).  The `PowerGridStateModel` 
detects this immediately — H2 rises from zero when the fault is introduced at t = 2.5 s.

This demonstrates the **diagnostic value** of joint phasor estimation over an 
independent-demodulator bank: the H2 channel correctly starts at 0 (even in the 
presence of the 230 V fundamental) and only rises when there is genuine H2 content.

In [ ]:
T_fault = 5.0
t_f = np.arange(int(T_fault * fs)) / fs
fault_onset = 2.5   # seconds: half-wave rectifier turns on
V2_fault = 20.0     # 2nd harmonic injected by the rectifier [V]

np.random.seed(7)
v_fault = (A1 * np.cos(2*np.pi*f0*t_f) +
           A3 * np.cos(2*np.pi*3*f0*t_f + phi3) +
           A5 * np.cos(2*np.pi*5*f0*t_f + phi5) +
           np.random.normal(0, noise_std, len(t_f)))
# Inject H2 after fault_onset
idx_on = int(fault_onset * fs)
v_fault[idx_on:] += V2_fault * np.cos(2*np.pi*2*f0*t_f[idx_on:])

grid_f = PowerGridStateModel("L1_fault", f0=f0, harmonics=5, fs=fs,
                              measurement_uncertainty=noise_std,
                              process_uncertainty=0.1)   # faster adaptation
grid_f.initial_state([[A1, 0], [0, 0], [A3, 0], [0, 0], [A5, 0]])
rec_f = grid_f.filter(v_fault, smooth=False, reset=True)   # causal (no smoother) to see detection speed

amp_f, sig_f = grid_f.harmonic_envelope(rec_f)

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.suptitle("Fault diagnosis: H2 rises when half-wave rectifier turns on at t=2.5 s", fontweight="bold")

colors_h = plt.cm.tab10(np.arange(5) % 10)
for k in range(5):
    axes[0].plot(t_f, amp_f[:, k], color=colors_h[k], lw=1.5, label=f"H{k+1}")
    axes[0].fill_between(t_f, amp_f[:, k] - sig_f[:, k], amp_f[:, k] + sig_f[:, k],
                         color=colors_h[k], alpha=0.18)
axes[0].axvline(fault_onset, ls="--", color="red", lw=1.5, label="fault onset")
axes[0].axhline(V2_fault, ls=":", color="darkorange", lw=1.2, label=f"true H2 = {V2_fault} V")
axes[0].set_ylabel("Amplitude (V)"); axes[0].set_title("Per-harmonic amplitudes")
axes[0].legend(fontsize=8, ncol=3); axes[0].set_ylim(bottom=0)

# Focus on H2 only
axes[1].plot(t_f, amp_f[:, 1], color="crimson", lw=2, label="H2 amplitude")
axes[1].fill_between(t_f, amp_f[:, 1] - sig_f[:, 1], amp_f[:, 1] + sig_f[:, 1],
                     color="crimson", alpha=0.2, label="±1σ")
axes[1].axvline(fault_onset, ls="--", color="red", lw=1.5, label="fault onset")
axes[1].axhline(V2_fault, ls=":", color="gray", lw=1.2, label=f"true H2 = {V2_fault} V")
axes[1].axhline(0, color="gray", lw=0.5)
axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("H2 amplitude (V)")
axes[1].set_title("H2 channel: zero before fault, detects rectifier load within ~0.5 s")
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

# Detection latency
threshold = V2_fault * 0.5
detect_idx = np.where((t_f > fault_onset) & (amp_f[:, 1] > threshold))[0]
if detect_idx.size:
    print(f"H2 crosses {threshold:.0f} V threshold at t = {t_f[detect_idx[0]]:.3f} s "
          f"(detection latency: {t_f[detect_idx[0]] - fault_onset:.3f} s)")
else:
    print("H2 did not cross threshold in this run — try increasing process_uncertainty.")

---
## Possible extensions

### 1. Voltage + current → complex power per harmonic
Add a second `PowerGridStateModel` for the current.  Active power $P_k = \tfrac{1}{2}\mathrm{Re}[V_k I_k^*]$, 
reactive power $Q_k = \tfrac{1}{2}\mathrm{Im}[V_k I_k^*]$ — computed per harmonic from the Kalman estimates.

### 2. Fundamental frequency tracking
Augment the state with a frequency offset $\delta f$ (random walk).  Update all $\omega_k$ 
dynamically — a **frequency-adaptive** harmonic tracker for weak grids.

### 3. Inter-harmonics
Arc furnaces and cycloconverters inject frequencies that are *not* integer multiples of $f_0$ 
(e.g. 75 Hz).  Add `PhasorOscillatorModel` entries at non-integer frequencies alongside the 
harmonic bank.

### 4. Real-time streaming
The `filter()` method processes a batch.  For causal streaming:
```python
for z_t in voltage_stream:
    kf._predict(kf.x, kf.P, u=None, t=t)
    kf._update(z_t, t=t)
    h1_amp = np.hypot(kf.x[0], kf.x[1])   # H1 amplitude in real time
    t += 1 / fs
```
This gives sub-cycle latency — far below DFT-based methods that need a full window.